# 01a Unpaved en Ernst input generatie
In dit script wordt de input gegenereerd die nodig is om de modellen te bouwen en te runnen.

In [46]:
from pathlib import Path
import numpy as np
import geopandas as gpd
import pandas as pd
from pathlib import Path
%matplotlib inline
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import contextily as cx
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

In [47]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [48]:
# Gevraagd wordt een dataframe voor de UNPAVED zoals dit:
# 	    id	    total_area	lu_areas	surface_level	soiltype	surface_storage	infiltration_capacity	initial_gwd	meteo_area	px	py	boundary_node
# code												
# 15.0	15.0	1375	250 0 0 0 0 0 0 0 0 0 225 0 0 0 0 0	                16.93   107	10.000	100.000	1.20	15.0	199378	395163	lat_15.0
# 55.0	55.0	303875	124200 18000 0 0 0 0 0 0 0 150 68125 0 11875 0...	21.69	105	10.000	100.000	1.20	55.0	197488	392239	lat_55.0
# 56.0	56.0	13300	5400 0 0 0 0 0 0 0 0 0 4425 0 375 0 1150 0	        20.63	113	10.000	100.000	1.20	56.0	197789	392200	lat_56.0
# 57.0	57.0	60925	6550 725 0 22800 0 0 0 0 0 0 9375 0 4600 0 175 0	21.49	113	10.000	100.000	1.20	57.0	197982	392247	lat_57.0

# en een dataframe voor ernst zoals deze:
# 	    id	    cvo	            lv	        cvi	    cvs
# code					
# 15.0	15.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00
# 55.0	55.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00
# 56.0	56.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00
# 57.0	57.0	300 2000 100000	0.0 1.0 2.0	300.00	5.00

### Basis settings

#### Selecteer simulatieperiode

In [49]:
# LONG RUN
# start_date = "2010-4-1"
# end_date = "2018-12-31"
# seizoenen = ["zomer", "winter"]
# date_range = pd.date_range(start_date, end_date, freq="2D")

# # LONG TEST
# start_date = "2014-07-1"
# end_date = "2016-09-30"
# date_range = pd.date_range(start_date, end_date, freq="3MS")
# seizoenen = ["zomer", "winter", "winter", "zomer"]

# SMALL TEST
# start_date = "2012-4-1"
# end_date = "2012-4-12"
# seizoenen = ["zomer", "winter"]
# date_range = pd.date_range(start_date, end_date, freq="2D")

# SIMULATIES TBV ONDERZOEK INLOOPTIJD ICM INITIËLE GRONDWATERSTAND

# # run_name, scenario = "OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930", "REF"
# start_date = "2014-01-01"
# end_date = "2016-09-30"
# seizoenen = WORDT VERDER NIET GEBRUIKT IN SCRIPT!
# date_range = WORDT VERDER NIET GEBRUIKT IN SCRIPT!

# # run_name, scenario = "OY_5_kD20_L4_W_InfMax_InitGWS_20130101_20160930", "REF"
# start_date = "2013-01-01"
# end_date = "2016-09-30"

# run_name, scenario = "OY_5_kD20_L4_W_InfMax_InitGWS_20120401_20160930", "REF"
start_date = "2012-04-01"
end_date = "2016-09-30"

In [50]:
# run_names = [
#     "OY_0_kD5_L4",
#     "OY_1_kD20_L4",
#     "OY_2_kD20_L2",
#     "OY_3_kD20_L4_W",
#     "OY_4_kD20_L4_W_InfMax",
#     "OY_5_kD20_L4_W_InfMax_InitGWS",
# ]

# # run_name, scenario = "OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930", "REF"
# run_names = ["OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930"]

# # run_name, scenario = "OY_5_kD20_L4_W_InfMax_InitGWS_20130101_20160930", "REF"
# run_names = ["OY_5_kD20_L4_W_InfMax_InitGWS_20130101_20160930"]

# run_name, scenario = "OY_5_kD20_L4_W_InfMax_InitGWS_20120401_20160930", "REF"
run_names = ["OY_5_kD20_L4_W_InfMax_InitGWS_20120401_20160930"]

#### Definieer pad naar data en input mappen

In [51]:
# INPUT vanuit WRIJ voor RR unpaved methode

main_dir = Path("D:\\153961_Oude_IJssel")
dir_data = Path(main_dir, "WRIJ_RR_Unpaved_methode_01_data")
dir_input = Path(main_dir, "WRIJ_RR_Unpaved_methode_02_input")

### Inlezen basisdata

- projectgebieden
- Watergangen
- Afwateringseenheden

In [52]:
dir_input_basis_data = dir_input / "basisdata"

project_areas_path = dir_input_basis_data / "gebieden.gpkg"
path_watergang = dir_input_basis_data / "watergang.gpkg"
path_afwateringseenheden = dir_input_basis_data / "afwateringseenheden.gpkg"

project_areas = gpd.read_file(project_areas_path, layer="gebieden")
watergang = gpd.read_file(path_watergang)
afwateringseenheden = gpd.read_file(path_afwateringseenheden)

Koppel de meteostations aan de afwateringseenheden

In [53]:
#Functie om de geodataframe aan te maken voor de zomer en de wintersituaties en het vullen van de geodataframe
def prepare_rr_input(df, koppeltabel):
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["xcoor"], df["ycoor"]),
        crs="EPSG:28992"
    )
    gdf = gdf.merge(
        koppeltabel[["GFEIDENT", "txt_file"]].rename(columns={"GFEIDENT": "GFEIDENT_old"}),
        on="GFEIDENT_old",
        how="left"
    )
    gdf["MeteoStationName"] = gdf["txt_file"].str.replace(".txt", "", regex=False).ffill().bfill()
    gdf.drop(columns="txt_file", inplace=True)
    return gdf

#### Functie om de unpaved dataframe en de ernst dataframe te genereren

Hier wordt een functie gedefinieerd om de geodataframe input om te schrijven naar de unpaved en ernst dataframe in het benodigde format.

In [54]:
def generate_unpaved_df_from_rr_input(gdf):
    df_unpaved = pd.DataFrame()
    df_unpaved["code"] = gdf["ID_RR_KNOOP"]    #"unpaved_" + gdf["GFEIDENT"]
    df_unpaved["id"] = gdf["ID_RR_KNOOP"]      #"unpaved_" + gdf["GFEIDENT"]
    df_unpaved["total_area"] = gdf["Area_RR_unpaved_m2"].astype(int)
    
    def list_area_per_land_use(total_area, croptype_index):
        lu_areas = [int(total_area) if i+1 == croptype_index else 0 for i in range(16)]
        return " ".join([str(a) for a in lu_areas])
    
    df_unpaved["lu_areas"] = gdf.apply(lambda x: list_area_per_land_use(x["Area_RR_unpaved_m2"], x["CropType"]), axis=1)
    df_unpaved["surface_level"] = gdf["SurfaceLevel_mNAP"]
    df_unpaved["soiltype"] = gdf["CapSimSoilType"] + 100 # soiltype_mapping + 100 # nog naar kijken of de code klopt
    df_unpaved["surface_storage"] = gdf["StorageOnLand_mm"]
    df_unpaved["infiltration_capacity"] = gdf["InfiltrationCapacity_mmph"]
    df_unpaved["initial_gwd"] = gdf["InitialGroundwaterLevel_mBelowSurface"]
    df_unpaved["meteo_area"] = gdf["MeteoStationName"]      #Kijken in koppeltabel
    df_unpaved["layer_thickness"] = gdf["LayerThickness_m"]
    df_unpaved["px"] = gdf.geometry.x
    df_unpaved["py"] = gdf.geometry.y
    df_unpaved["boundary_node"] = "lat_" + gdf["ID_RR_KNOOP"].astype(str)
    df_unpaved["boundary_waterlevel"] = gdf["OpenWaterLevelBoundary_mNAP"]
    df_unpaved = df_unpaved.set_index("code")
    return df_unpaved


def generate_ernst_df_from_rr_input(gdf, max_drainage_value=9999999):
    for col in ["FirstDrainResistance_d", "SecondDrainResistance_d", "ThirdDrainResistance_d", "OpenWaterHorizontalInflowResistance_d", "SurfaceOverlandFlowResistance_d"]:
        gdf.loc[gdf[col]>max_drainage_value, col] = max_drainage_value
        gdf[col] = gdf[col].round(3)
    df_ernst = pd.DataFrame()
    df_ernst["code"] = gdf["ID_RR_KNOOP"]      # "ernst_" + gdf["GFEIDENT"]
    df_ernst["id"] = gdf["ID_RR_KNOOP"]        # "ernst_" + gdf["GFEIDENT"]

    gdf["FirstDrainLevel_m"] = round(gdf["SurfaceLevel_mNAP"] - gdf["FirstDrainLevel_mNAP"],2)
    gdf["SecondDrainLevel_m"] = round(gdf["SurfaceLevel_mNAP"] - gdf["SecondDrainLevel_mNAP"],2)
    gdf["ThirdDrainLevel_m"] = round(gdf["SurfaceLevel_mNAP"] - gdf["ThirdDrainLevel_mNAP"],2)

    def generate_weerstanden_levels(row):
        drainweerstanden = list(row[["FirstDrainResistance_d", "SecondDrainResistance_d", "ThirdDrainResistance_d"]])
        drainlevels = list(row[["FirstDrainLevel_m", "SecondDrainLevel_m", "ThirdDrainLevel_m"]])
        drainweerstanden_str = []
        drainlevels_str = []
        for w, l in zip(drainweerstanden, drainlevels):
            if l == 0.0 or l in drainlevels_str:
                drainlevels_str.insert(0, "0")
                drainweerstanden_str.insert(0, "0")
            else:
                drainlevels_str.append(f"{l:0.3f}")
                drainweerstanden_str.append(f"{w:0.3f}")
        openwaterdrainweerstand = row["OpenWaterDrainResistance_d"]
        drainweerstanden_str.append(f"{openwaterdrainweerstand:0.3f}")
        # VERWIJDER EERSTE WEERSTAND. IN DRRWRITER WORDT VOORAAN EEN 0 GEZET (GEEN IDEE WAAROM?)
        return " ".join(drainweerstanden_str[1:]), " ".join(drainlevels_str)
    
    df_ernst[["cvo", "lv"]] = gdf.apply(
        lambda row: generate_weerstanden_levels(row), 
        axis=1,
        result_type="expand"
    )
    df_ernst["cvi"] = gdf["OpenWaterHorizontalInflowResistance_d"].astype(str) # 10 * een 9
    df_ernst["cvs"] = gdf["SurfaceOverlandFlowResistance_d"].astype(str)
    df_ernst = df_ernst.set_index("code")
    return df_ernst


def generate_rr_unpaved_ernst_from_input(dir_scenario_input, dir_scenario_output):
    if not Path(dir_scenario_output).exists():
        Path(dir_scenario_output).mkdir(parents=True)
    seasons = ["zomer", "winter"]
    gpkg_rr_input_zomer_winter = ["RR_input_ZOMER.gpkg", "RR_input_WINTER.gpkg"]

    gdf_rr_input = {}

    for season, gpkg_rr_input in zip(seasons, gpkg_rr_input_zomer_winter):
        print(f"   - season: {season}")

        gdf = gpd.read_file(dir_scenario_input / gpkg_rr_input)
        gdf = gdf[~gdf["ID_RR_KNOOP"].duplicated()]

        df_ernst = generate_ernst_df_from_rr_input(gdf)
        df_unpaved = generate_unpaved_df_from_rr_input(gdf)
        gdf_unpaved = gpd.GeoDataFrame(df_unpaved, geometry=gpd.points_from_xy(df_unpaved.px, df_unpaved.py), crs=28992)

        gdf_rr_input[season] = {}
        gdf_rr_input[season]["input"] = gdf
        gdf_rr_input[season]["ernst"] = df_ernst
        gdf_rr_input[season]["unpaved"] = gdf_unpaved

        df_ernst.to_csv(dir_scenario_output / f"df_ernst_{season}.csv")
        df_unpaved.to_csv(dir_scenario_output / f"df_unpaved_{season}.csv")
        gdf_unpaved.to_file(dir_scenario_output / f"gdf_unpaved_{season}.gpkg", layer=f"gdf_unpaved_{season}", driver="GPKG")
    
    return gdf_rr_input   

## Genereer input

Hier wordt de input voor de modellen gegeneerd voor de referentie en het scenario. Eerst voor het hele oude IJssel gebied, en later opgesplitst per pilot gebied.

In [55]:
dir_input

WindowsPath('D:/153961_Oude_IJssel/WRIJ_RR_Unpaved_methode_02_input')

Genereer een aparte inputmap per pilotgebied

In [56]:
for run_name in run_names:
    # AREAS
    for i, area in project_areas.iterrows():
        dir_input_area = dir_input / "rr_input_area_scenario" / run_name / f"gebied_{area.area_id}"
        dir_input_area.mkdir(parents=True, exist_ok=True)

        area_gdf = project_areas.iloc[[i]]
        area_gdf.to_file(dir_input_area / f"gebied.gpkg", layer=f"gebied", driver="GPKG")

#### Genereer input voor het hele Oude IJssel gebied

Hier wordt de input gegenereerd voor het hele Oude IJssel gebied. 
Elke RR knoop wordt aan een afwateringseenheid gekoppeld.
De laterale knopen worden aangemaakt.

En de input wordt weggeschreven per scenario en seizoen.

In [57]:
# INPUT vanuit WRIJ voor RR unpaved methode
dir_data_scenarios = Path(dir_input, "rr_data_scenarios")
dir_input_scenarios = Path(dir_input, "rr_input_scenarios")

run_scenario_gdf_input = {}

for run_name in run_names:
    print(f"Run: {run_name}")
    scenario_gdf_input = {}
    list_scenarios = [p.name for p in list(Path(dir_data_scenarios, run_name).iterdir())]

    for scenario in list_scenarios:
        print(f" * Scenario: {scenario}")
        dir_scenario_input = dir_data_scenarios / run_name / scenario
        dir_scenario_output = dir_input_scenarios / run_name / scenario

        # inlezen input
        path_input_scen_zomer = dir_scenario_input / f"RRunpaved_KNOPEN_{scenario}_ZOMER.csv"
        path_input_scen_winter = dir_scenario_input / f"RRunpaved_KNOPEN_{scenario}_WINTER.csv"

        input_scen_zomer = pd.read_csv(path_input_scen_zomer, delimiter=',')
        input_scen_winter = pd.read_csv(path_input_scen_winter, delimiter=',')
        input_scen_zomer["GFEIDENT_old"] = input_scen_zomer["GFEIDENT"]
        input_scen_winter["GFEIDENT_old"] = input_scen_winter["GFEIDENT"]

        scen_zomer = gpd.GeoDataFrame(input_scen_zomer, geometry=gpd.points_from_xy(input_scen_zomer.xcoor, input_scen_zomer.ycoor, crs=28992))
        scen_winter = gpd.GeoDataFrame(input_scen_winter, geometry=gpd.points_from_xy(input_scen_winter.xcoor, input_scen_winter.ycoor, crs=28992))
        scen_zomer = scen_zomer.drop(columns=["GFEIDENT"]).sjoin(afwateringseenheden[["GFEIDENT", "geometry"]], how="left")
        scen_winter = scen_winter.drop(columns=["GFEIDENT"]).sjoin(afwateringseenheden[["GFEIDENT", "geometry"]], how="left")
        
        input_scen_zomer = scen_zomer.drop(columns="geometry")
        input_scen_winter = scen_winter.drop(columns="geometry")
        
        #inladen koppeltabel
        path_koppeltabel = Path(dir_data_scenarios, "meteo\\neerslag_tijdreeksen\\output_koppeltabel\\koppeltabel.csv")
        koppeltabel = pd.read_csv(path_koppeltabel, delimiter=",")
        
        # debug: check for duplicates in koppeltabel
        koppeltabel = koppeltabel.drop_duplicates(subset=["GFEIDENT"], keep="first")

        #wegschrijven RR input
        rr_input_zomer = prepare_rr_input(input_scen_zomer, koppeltabel)
        rr_input_winter = prepare_rr_input(input_scen_winter, koppeltabel)
        
        afwateringseenheden_laterals = afwateringseenheden.merge(rr_input_zomer[["GFEIDENT", "ID_RR_KNOOP"]], on="GFEIDENT", how="right")

        afwateringseenheden_laterals["code"] = afwateringseenheden_laterals["ID_RR_KNOOP"].astype(str)
        afwateringseenheden_laterals["globalid"] = afwateringseenheden_laterals["GLOBALID"].astype(str)
        afwateringseenheden_laterals["lateraleknoopid"] = "lat_" + afwateringseenheden_laterals["ID_RR_KNOOP"].astype(str)
        afwateringseenheden_laterals = afwateringseenheden_laterals[["code", "globalid", "lateraleknoopid", "geometry"]]
        afwateringseenheden_laterals = afwateringseenheden_laterals[~afwateringseenheden_laterals.geometry.isnull()]

        afwateringseenheden_laterals.to_file(dir_scenario_input / "afwateringseenheden_laterals.gpkg", driver="GPKG")

        list_rr_nodes_codes = list(afwateringseenheden_laterals["code"].values)

        rr_input_zomer = rr_input_zomer[rr_input_zomer["ID_RR_KNOOP"].isin(list_rr_nodes_codes)]
        rr_input_winter = rr_input_winter[rr_input_winter["ID_RR_KNOOP"].isin(list_rr_nodes_codes)]
        
        rr_input_zomer.to_file(dir_scenario_input / "RR_input_ZOMER.gpkg", driver="GPKG")
        rr_input_winter.to_file(dir_scenario_input / "RR_input_WINTER.gpkg", driver="GPKG")
        
        gdfs_rr_input = generate_rr_unpaved_ernst_from_input(dir_scenario_input, dir_scenario_output)
        scenario_gdf_input[scenario] = gdfs_rr_input

    run_scenario_gdf_input[run_name] = scenario_gdf_input

Run: OY_5_kD20_L4_W_InfMax_InitGWS_20120401_20160930
 * Scenario: REF
   - season: zomer
   - season: winter


#### Clip de gegenereerde input per pilot gebied

Hier wordt de gegenereerde modelinput per pilot gebied geknipt en weggeschreven per scenario/seizoen.

In [58]:
for run_name in run_names:
    print(f"Run: {run_name}")
    scenario_gdf_input = run_scenario_gdf_input[run_name]

    list_scenarios = list(scenario_gdf_input.keys())
    seasons = scenario_gdf_input[list_scenarios[0]].keys()

    for i, project_area in project_areas.iterrows():
        print(f" * Gebied {project_area.area_id}")
        dir_input_area = dir_input / "rr_input_area_scenario" / run_name / f"gebied_{project_area.area_id}"
        if not dir_input_area.exists():
            dir_input_area.mkdir(parents=True, exist_ok=True)
        
        for scenario in list_scenarios:
            print(f"   - {scenario}")
            dir_input_area_scenario = dir_input_area / scenario
            if not dir_input_area_scenario.exists():
                dir_input_area_scenario.mkdir(parents=True, exist_ok=True)
            
            for season in seasons:
                # print("     . {season}")
                # rr_unpaved
                scenario_gdf_unpaved_area = scenario_gdf_input[scenario][season]["unpaved"].clip(project_area.geometry)
                scenario_gdf_unpaved_area.to_file(
                    dir_input_area_scenario / f"gdf_unpaved_{season}.gpkg", 
                    layer=f"gdf_unpaved_{season}", 
                    driver="GPKG"
                )

                # rr_unpaved
                scenario_df_unpaved_area = scenario_gdf_unpaved_area.drop(columns="geometry")
                scenario_df_unpaved_area.to_csv(dir_input_area_scenario / f"df_unpaved_{season}.csv")

                # rr_ernst
                sel_ernst = scenario_gdf_input[scenario][season]["ernst"].index.str.replace("ernst_", "unp_")
                scenario_df_ernst_area = scenario_gdf_input[scenario][season]["ernst"].loc[sel_ernst.isin(scenario_gdf_unpaved_area.index)]
                scenario_df_ernst_area.to_csv(dir_input_area_scenario / f"df_ernst_{season}.csv")

        list_rr_nodes_codes = list(scenario_gdf_unpaved_area.index.str.replace("unp_", ""))
        
        # afwateringseenheden
        afwateringseenheden_laterals = gpd.read_file(dir_scenario_input / "afwateringseenheden_laterals.gpkg")
        afwateringseenheden_laterals_area = afwateringseenheden_laterals[afwateringseenheden_laterals["code"].isin(list_rr_nodes_codes)]
        afwateringseenheden_laterals_area.to_file(dir_input_area / f"afwateringseenheden.gpkg", layer=f"afwateringseenheden", driver="GPKG")

        # watergang
        watergang_area = watergang.clip(project_area.geometry).explode()
        watergang_area.to_file(dir_input_area / f"watergang.gpkg", layer=f"watergang", driver="GPKG")

        # kwel/wegzijging (seepage)
        seepage_area = pd.DataFrame(
            columns=["sep_" + afw_eenheid for afw_eenheid in list_rr_nodes_codes],
            index=pd.date_range(start=start_date, end=end_date, freq="MS")
        ).fillna(0.0)
        seepage_area.to_csv(dir_input_area / f"seepage.csv")

Run: OY_5_kD20_L4_W_InfMax_InitGWS_20120401_20160930
 * Gebied 0
   - REF
 * Gebied 1
   - REF
 * Gebied 2
   - REF
 * Gebied 3
   - REF


### METEO DATA: PRECIPITATION AND EVAPORATION (ALLES)

Pas de start_date en end_date aan zodat die alles omvat

In [59]:
start_date = pd.Timestamp(start_date) - pd.Timedelta(days=35)
end_date = pd.Timestamp(end_date) + pd.Timedelta(days=35)

functie om bui-file weg te schrijven

In [60]:
from datetime import timedelta

def write_bui(df, outfile, timestep_seconds=3600):
    stations = list(df.columns)
    nstations = len(stations)

    start = df.index[0]
    end = df.index[-1]

    duration = end - start + timedelta(seconds=timestep_seconds)

    dd = duration.days
    hh, rem = divmod(duration.seconds, 3600)
    mm, ss = divmod(rem, 60)

    with open(outfile, "w") as f:
        # Header
        f.write(f"*Name of this file: {outfile}\n")
        f.write("*Date and time of construction: 00/00/2000 00:00:00.\n")
        f.write("1\n")
        f.write("*Aantal stations\n")
        f.write(f"{nstations}\n")
        f.write("*Namen van stations\n")

        for s in stations:
            f.write(f"'{s}'\n")

        f.write("*Aantal gebeurtenissen (omdat het 1 bui betreft is dit altijd 1)\n")
        f.write("*en het aantal seconden per waarnemingstijdstap\n")
        f.write(f"1 {timestep_seconds}\n")
        f.write("*Elke commentaarregel wordt begonnen met een * (asterisk).\n")
        f.write("*Eerste record bevat startdatum en -tijd, lengte van de gebeurtenis in dd hh mm ss\n")
        f.write("*Het format is: yyyymmdd:hhmmss:ddhhmmss\n")
        f.write("*Daarna voor elk station de neerslag in mm per tijdstap.\n")

        # Startrecord
        f.write(
            f"{start.year} {start.month} {start.day} "
            f"{start.hour} {start.minute} {start.second} "
            f"{dd} {hh} {mm} {ss}\n"
        )

        # Tijdstappen
        for _, row in df.iterrows():
            line = " ".join(f"{v:.3f}" for v in row.values)
            f.write(line + "\n")

Hier worden de neerslagtijdreeksen per meteostation ingelezen en weggeschreven als .BUI bestand.

In [61]:
# METEO - Neerslag
dir_neerslag_data = Path(dir_data_scenarios, "meteo", "neerslag_tijdreeksen\\output_tijdreeksen")

# select meteostations from scenario input
meteo_stations = scenario_gdf_input[list_scenarios[0]]["zomer"]["input"]["MeteoStationName"].unique()
neerslag_tijdseries = pd.DataFrame(
    columns=["ms_" + ms for ms in meteo_stations]
)

for meteo_station in meteo_stations:
    path_tijdserie = Path(dir_neerslag_data, meteo_station + ".txt")
    if path_tijdserie.exists():
        tijdserie = pd.read_csv(path_tijdserie, sep=";", index_col=0, parse_dates=["YYYYMMDDHH"], date_format="%Y%m%d%H")
        neerslag_tijdseries["ms_" + meteo_station] = tijdserie

In [62]:
neerslag_tijdseries = neerslag_tijdseries.loc[start_date:end_date]
write_bui(
    neerslag_tijdseries, 
    Path(dir_input_scenarios, "meteo", "METEO_NEERSLAG.BUI"), 
    timestep_seconds=3600
)

Hier wordt de verdampingstijdreeks ingelezen en weggeschreven als .EVP bestand. 

In [63]:
# METEO - Verdamping
dir_meteo = Path(dir_data_scenarios, "meteo")
file_evp = "verdamping_hupsel.xlsx"

evp = pd.read_excel(dir_meteo / file_evp, index_col=0, parse_dates=True)
evp.columns = ["evp"]

evp["jaar"] = evp.index.year
evp["maand"] = evp.index.month
evp["dag"] = evp.index.day
evp = evp[["jaar", "maand", "dag", "evp"]]

evp = evp.loc[start_date:end_date]

header_evp_file = (
    "*evpsfile Verdamping Hupsel\n"
    "*Meteo data: evaporation intensity in mm/day\n"
    "*First record: start date, data in mm/day\n"
    "*Datum (year month day), evp (mm/dag) voor elk weerstation\n"
    "*jaar maand dag evp[mm]\n"
)

output_file = Path(dir_input_scenarios, "meteo", "METEO_verdamping.EVP")

with open(output_file, "w") as f:
    f.write(header_evp_file)
    evp.to_string(
        f,
        index=False,
        header=False,
        formatters={"evp": "{:.3f}".format}
    )

# evp

#### METEO: NEERSLAG EN VERDAMPING (SPECIFIEK PER GEBIED)

In [64]:
list_scenarios = list(scenario_gdf_input.keys())
seasons = scenario_gdf_input[list_scenarios[0]].keys()

Neerslag per pilotgebied wegschrijven.

In [65]:
# METEO - Neerslag
dir_neerslag_data = Path(dir_data_scenarios, "meteo", "neerslag_tijdreeksen\\output_tijdreeksen")

# select meteostations from scenario input
neerslag_tijdseries = neerslag_tijdseries.loc[start_date:end_date]

for run_name in run_names:
    print(f"Run: {run_name}")
    for i, area in project_areas.iterrows():
        print(f" * Gebied {area.area_id}")
        dir_input_area = dir_input / "rr_input_area_scenario" / run_name / f"gebied_{area.area_id}"
        dir_input_area_meteo = dir_input_area / "meteo"
        if not dir_input_area_meteo.exists():
            dir_input_area_meteo.mkdir(parents=True, exist_ok=True)

        for scenario in list_scenarios:
            for season in seasons:
                # meteo
                scenario_gdf_input_area = scenario_gdf_input[scenario][season]["input"].clip(area.geometry)
                meteo_stations_area = list(scenario_gdf_input_area["MeteoStationName"].unique())
                meteo_stations_area = ["ms_" + ms for ms in meteo_stations_area]
                precipitation_timeseries_area = neerslag_tijdseries[meteo_stations_area]
                write_bui(
                    precipitation_timeseries_area, 
                    dir_input_area_meteo / "METEO_NEERSLAG.BUI", 
                    timestep_seconds=3600
                )
                break
            break

Run: OY_5_kD20_L4_W_InfMax_InitGWS_20120401_20160930
 * Gebied 0
 * Gebied 1
 * Gebied 2
 * Gebied 3


Verdamping per pilot gebied wegschrijven.

In [66]:
# # METEO - Verdamping
dir_meteo = Path(dir_data_scenarios, "meteo")
file_evp = "verdamping_hupsel.xlsx"

evp = pd.read_excel(dir_meteo / file_evp, index_col=0, parse_dates=True)
evp.columns = ["evp"]

evp["jaar"] = evp.index.year
evp["maand"] = evp.index.month
evp["dag"] = evp.index.day
evp = evp[["jaar", "maand", "dag", "evp"]]

evp = evp.loc[start_date:end_date]

header_evp_file = (
    "*evpsfile Verdamping Hupsel\n"
    "*Meteo data: evaporation intensity in mm/day\n"
    "*First record: start date, data in mm/day\n"
    "*Datum (year month day), evp (mm/dag) voor elk weerstation\n"
    "*jaar maand dag evp[mm]\n"
)

for run_name in run_names:
    print(f"Run: {run_name}")
    for i, area in project_areas.iterrows():
        print(f" * Gebied {area.area_id}")
        dir_input_area = dir_input / "rr_input_area_scenario" / run_name / f"gebied_{area.area_id}"
        dir_input_area_meteo = dir_input_area / "meteo"
        if not dir_input_area_meteo.exists():
            dir_input_area_meteo.mkdir(parents=True, exist_ok=True)

        evp_file = Path(dir_input_area_meteo, "METEO_VERDAMPING.EVP")

        with open(evp_file, "w") as f:
            f.write(header_evp_file)
            evp.to_string(
                f,
                index=False,
                header=False,
                formatters={"evp": "{:.3f}".format}
            )

Run: OY_5_kD20_L4_W_InfMax_InitGWS_20120401_20160930
 * Gebied 0
 * Gebied 1
 * Gebied 2
 * Gebied 3


In [67]:
evp_file

WindowsPath('D:/153961_Oude_IJssel/WRIJ_RR_Unpaved_methode_02_input/rr_input_area_scenario/OY_5_kD20_L4_W_InfMax_InitGWS_20120401_20160930/gebied_3/meteo/METEO_VERDAMPING.EVP')